# Biotic Hardware Synthesis — Comparative Morphology Demo

Runs the full deterministic pipeline and compares fractal, botanical, and random control morphologies.

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, 'run.py'], capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_f = pd.read_csv('data/simulation_results_fractal.csv')
df_b = pd.read_csv('data/simulation_results_botanical.csv')
df_r = pd.read_csv('data/simulation_results_random.csv')

def norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-12)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_f['Distance'], norm(df_f['Merit_Scaled']),    label='Fractal - Merit Scaled',   linewidth=2, color='#2196F3')
ax.plot(df_f['Distance'], norm(df_f['Coherence_Ratio']), label='Fractal - Coherence',       linestyle='--', color='#2196F3', alpha=0.6)
ax.plot(df_b['Distance'], norm(df_b['Merit_Scaled']),    label='Botanical - Merit Scaled',  linewidth=2, color='#4CAF50')
ax.plot(df_b['Distance'], norm(df_b['Coherence_Ratio']), label='Botanical - Coherence',     linestyle='--', color='#4CAF50', alpha=0.6)
ax.plot(df_r['Distance'], norm(df_r['Merit_Scaled']),    label='Random - Merit Scaled',     linewidth=2, color='#FF5722')
ax.plot(df_r['Distance'], norm(df_r['Coherence_Ratio']), label='Random - Coherence',        linestyle='--', color='#FF5722', alpha=0.6)

ax.set_xlabel('Distance')
ax.set_ylabel('Normalized Metrics')
ax.set_title('Morphological Sensitivity Benchmark v1.1 - Bio-Inspired vs. Random Control')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Statistical Separation Analysis

Welch's t-test and Cohen's d across morphology pairs on `Merit_Scaled`.

- **p < 0.05**: the two morphologies are statistically distinguishable
- **Cohen's d**: effect size — small (<0.5), medium (0.5–0.8), large (>0.8)

In [ ]:
from math import lgamma, exp

def welch_ttest(a, b):
    a, b = np.array(a), np.array(b)
    n1, n2 = len(a), len(b)
    s1, s2 = np.var(a, ddof=1), np.var(b, ddof=1)
    t = (np.mean(a) - np.mean(b)) / np.sqrt(s1/n1 + s2/n2)
    df = (s1/n1 + s2/n2)**2 / ((s1/n1)**2/(n1-1) + (s2/n2)**2/(n2-1))
    x = df / (df + t**2)
    def beta_inc(a, b, x, steps=1000):
        if x <= 0: return 0.0
        if x >= 1: return 1.0
        lbeta = lgamma(a) + lgamma(b) - lgamma(a+b)
        result = sum(
            ((i+0.5)/steps * x)**(a-1) * (1 - (i+0.5)/steps * x)**(b-1) * (x/steps)
            for i in range(steps)
        )
        return result / exp(lbeta)
    p = 2 * beta_inc(df/2, 0.5, x)
    return float(t), min(float(p), 1.0)

def cohens_d(a, b):
    a, b = np.array(a), np.array(b)
    pooled = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return float((np.mean(a) - np.mean(b)) / (pooled + 1e-12))

metric = 'Merit_Scaled'
pairs = [
    ('Fractal',   df_f[metric], 'Botanical', df_b[metric]),
    ('Fractal',   df_f[metric], 'Random',    df_r[metric]),
    ('Botanical', df_b[metric], 'Random',    df_r[metric]),
]

rows = []
for n1, s1, n2, s2 in pairs:
    t, p = welch_ttest(s1, s2)
    d = cohens_d(s1, s2)
    sig  = 'yes' if p < 0.05 else 'no'
    size = 'large' if abs(d) > 0.8 else 'medium' if abs(d) > 0.5 else 'small'
    rows.append({'Pair': f'{n1} vs {n2}', 't': round(t,3), 'p': round(p,4),
                 'Significant (p<0.05)': sig, "Cohen's d": round(d,3), 'Effect size': size})

pd.DataFrame(rows).set_index('Pair')